# Chapter 7: Supervised Learning, Classification

**Foundations of Data Science**

## In this lesson

1. A classifier outputs a probability; the decision rule is a separate choice
2. Logistic regression: odds, log-odds, and what the coefficients mean
3. The accuracy trap, and the baseline you have to beat
4. A proper pipeline: split, preprocess inside, tune, evaluate once
5. Trees, forests, and boosting: flexibility, and the price of it
6. What the model learned: coefficients, importances, partial dependence
7. Choosing between models

## The one idea

**A classifier doesn't make decisions. It makes probabilities, and someone else chooses the cutoff.** The model says "this patient has a 23% chance of diabetes." Whether 23% is enough to order a test is a question about costs: a missed case versus an unnecessary test. Today we build the probability machine. Next chapter is about the cutoff and how to judge the machine.

The clinic question, finally in its natural form: **given what a screening visit measures (age, sex, weight, waist, blood pressure, income bracket), which patients should be flagged for a diabetes test?**

## Setup

<!-- nb-links -->
[Download this notebook](chapter_07_classification.ipynb?download=1){download="chapter_07_classification.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/chapter_07/chapter_07_classification.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Define the problem

Outcome: `diabetes` (0/1, self-reported doctor diagnosis). Predictors: things available at a screening visit, nothing from a lab. Adults 20+.

First number to know about any classification problem: the **base rate**. How common is the positive class? It sets the floor every model has to beat, and it decides which metrics mean anything.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["race_eth"] = adults["race_eth"].map({1: "Mexican American", 2: "Other Hispanic", 3: "White", 4: "Black", 6: "Asian", 7: "Other/multi"})

num = ["age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"]
cat = ["sex", "race_eth"]
X = adults[num + cat]
y = adults["diabetes"]

print("n =", len(y))
print("base rate:", y.mean().round(3), "   (so 'predict nobody' is", (1 - y.mean()).round(3), "accurate)")

## 2. Logistic regression

Linear regression predicts a number that can be anything. A probability has to stay between 0 and 1. Logistic regression gets there by modeling the **log-odds** as a line:

$$\log\frac{p}{1-p} = \beta_0 + \beta_1 x_1 + \dots$$

In words. **Odds** are p/(1-p): a 20% probability is odds of 0.25 (one to four). **Log-odds** are the log of that, which runs from minus infinity to plus infinity, so a line can model it. The model fits the line, then converts back: $p = 1/(1+e^{-z})$, the S-shaped curve.

The coefficients are read as **odds ratios** after exponentiating: $e^{\beta}$ is the factor the odds are multiplied by per unit of the predictor. An odds ratio of 1.5 for "per 10 years of age" means each decade multiplies the odds of diabetes by 1.5, holding everything else fixed. Same adjusted-comparison logic as chapter 6.

In [ ]:
import statsmodels.formula.api as smf

# Rescale age and BMI so the odds ratios are per decade and per 5 BMI units (per-year and per-unit ratios are tiny and hard to read)
lr_df = adults.assign(age10=adults["age"] / 10, bmi5=adults["bmi"] / 5).dropna(subset=["bmi"])
logit = smf.logit("diabetes ~ age10 + bmi5 + sex + C(race_eth, Treatment('White'))", data=lr_df).fit(disp=0)

ors = pd.DataFrame({"odds_ratio": np.exp(logit.params), "ci_low": np.exp(logit.conf_int()[0]), "ci_high": np.exp(logit.conf_int()[1]), "p": logit.pvalues}).round(3)
ors.index = [i.replace("C(race_eth, Treatment('White'))", "race:") for i in ors.index]
ors

Each decade of age multiplies the odds of diabetes by about 1.9; each 5 BMI points by about 1.5; men have about 1.5 times the odds of women at the same age and BMI.

The race/ethnicity rows need care. Asian adults have 2.3 times the adjusted odds of White adults, the highest of any group, and Mexican American adults about 2. Black adults come out at 1.25 with an interval touching 1.0, barely distinguishable from White adults. Compare that to the chapter 4 heatmap, where Asian adults looked unremarkable and Black adults looked elevated. What changed is BMI: Asian adults have lower BMI on average, so *at the same BMI* their risk is much higher; the opposite adjustment pulls the Black coefficient down. This is a known result (clinical guidelines use lower BMI cutoffs for Asian populations for exactly this reason), and it's a reminder that "adjusted for" changes the question being asked.

## 3. The accuracy trap

Fit the model, call anyone above 50% a positive, and measure accuracy. Then compare to a model that does nothing.

In [ ]:
from sklearn.metrics import accuracy_score

p_hat = logit.predict(lr_df)
pred_50 = (p_hat >= 0.5).astype(int)
print("logistic regression, threshold 0.5:")
print("  accuracy:", round(accuracy_score(lr_df['diabetes'], pred_50), 3))
print("  positives flagged:", pred_50.sum(), "of", lr_df['diabetes'].sum(), "actual cases")
print("\npredict nobody has diabetes:")
print("  accuracy:", round(accuracy_score(lr_df['diabetes'], np.zeros(len(lr_df))), 3))

84.3% accuracy, and it flagged 127 people out of 815 actual cases. The do-nothing model scores 84.2%. One tenth of a point separates a fitted model from no model at all. **Accuracy is meaningless when one class dominates.** The 0.5 cutoff is also wrong for this problem: with a 16% base rate, almost nobody's probability reaches 50%, so almost nobody gets flagged. Lower the cutoff and you catch more cases at the cost of more false alarms. Where to set it is next chapter's subject; for today, we judge models by a metric that doesn't depend on the cutoff.

**AUC** (area under the ROC curve), in one sentence: the probability that a randomly chosen case gets a higher score than a randomly chosen non-case. 0.5 is a coin flip; 1.0 is perfect ranking. It judges the probabilities, not any particular cutoff. Full treatment next chapter.

## 4. A proper pipeline

Same discipline as chapter 6: 60/20/20 split, preprocessing fit inside the pipeline on training rows only, the regularization strength picked on validation, test touched once. Two additions for classification: **stratify** the split so each part has the same base rate, and let `model_tuner` report the whole metric set.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from model_tuner import Model

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
])

def make_model(name, estimator, grid):
    return Model(
        name=name, estimator_name=name, estimator=estimator, model_type="classification",
        pipeline_steps=[("pre", preprocess)], grid=grid, scoring=["roc_auc"],
        stratify_y=True, train_size=0.6, validation_size=0.2, test_size=0.2,
        random_state=42, display=False,
    )

lr = make_model("lr", LogisticRegression(max_iter=2000), [{"lr__C": [0.01, 0.1, 1, 10]}])
lr.grid_search_param_tuning(X, y)
X_train, y_train = lr.get_train_data(X, y)
X_valid, y_valid = lr.get_valid_data(X, y)
X_test, y_test = lr.get_test_data(X, y)
lr.fit(X_train, y_train)

print("chosen C:", lr.best_params_per_score["roc_auc"]["params"])
print("base rate by split:", y_train.mean().round(3), y_valid.mean().round(3), y_test.mean().round(3))

In [ ]:
from model_metrics import summarize_model_performance

# Test-set metrics at the default 0.5 threshold. Read AUC and Average Precision; ignore the threshold-dependent rows until next chapter.
summarize_model_performance(model=lr.estimator, X=X_test, y=y_test, model_title="Logistic regression", return_df=True)

## 5. Trees, forests, boosting

Logistic regression draws one straight boundary in log-odds space. A **decision tree** asks a sequence of yes/no questions ("age > 45? BMI > 30?") and can carve out any shape. That flexibility is the point and the problem.

- A **single tree** is fully interpretable (you can draw it) and overfits badly if allowed to grow.
- A **random forest** fits hundreds of trees on random subsets of rows and columns and averages them. Averaging cancels out each tree's overfitting. Less interpretable, usually better.
- **Gradient boosting** (XGBoost, CatBoost, LightGBM) fits trees in sequence, each one correcting the last's mistakes. Often the strongest model on tabular data; needs more care with tuning.

First, a small tree you can read.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# A depth-3 tree on the preprocessed training data. Shallow on purpose: this is for reading, not for accuracy.
small_tree = Pipeline([("pre", preprocess), ("tree", DecisionTreeClassifier(max_depth=3, random_state=42))]).fit(X_train, y_train)
feature_names = small_tree["pre"].get_feature_names_out()

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(small_tree["tree"], feature_names=feature_names, class_names=["no", "diabetes"], filled=True, proportion=True, rounded=True, fontsize=9, ax=ax)
plt.show()

The first split is on age (standardized, so the number is in SD units). Then BMI or waist, then age again. The tree rediscovered the chapter 1 plot. Note the `value` fields: the share of positives in each leaf is the tree's probability estimate, which is why a small tree gives very coarse probabilities (only 8 distinct values).

Now the overfitting demonstration: let the tree grow without limit.

In [ ]:
from sklearn.metrics import roc_auc_score

for depth in [3, 6, None]:
    t = Pipeline([("pre", preprocess), ("tree", DecisionTreeClassifier(max_depth=depth, random_state=42))]).fit(X_train, y_train)
    auc_tr = roc_auc_score(y_train, t.predict_proba(X_train)[:, 1])
    auc_te = roc_auc_score(y_test, t.predict_proba(X_test)[:, 1])
    print(f"max_depth={str(depth):<5} train AUC {auc_tr:.3f}   test AUC {auc_te:.3f}   gap {auc_tr - auc_te:.3f}")

Unlimited depth: train AUC 1.0, test AUC far below the depth-3 tree. The tree memorized the training rows. This gap, train minus test, is the number to watch for every flexible model; it was near zero for ridge in chapter 6 and it's huge here.

Forests and boosting fix this by averaging or by sequential correction. Both have settings that control flexibility (`max_depth`, `min_samples_leaf`, `n_estimators`, learning rate), and those settings get chosen on the validation set, never on test.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = make_model("rf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
                [{"rf__max_depth": [4, 8, None], "rf__min_samples_leaf": [1, 10, 30]}])
xgb = make_model("xgb", XGBClassifier(n_estimators=300, random_state=42, n_jobs=-1, eval_metric="logloss"),
                 [{"xgb__max_depth": [2, 3, 4], "xgb__learning_rate": [0.03, 0.1], "xgb__subsample": [0.8]}])

for m in (rf, xgb):
    m.grid_search_param_tuning(X, y)
    m.fit(X_train, y_train)
    print(m.name, "chosen:", m.best_params_per_score["roc_auc"]["params"])

In [ ]:
# Three models, same test set, cutoff-free comparison
rows = []
for m, label in [(lr, "Logistic regression"), (rf, "Random forest"), (xgb, "XGBoost")]:
    rows.append({
        "model": label,
        "train AUC": roc_auc_score(y_train, m.predict_proba(X_train)[:, 1]),
        "valid AUC": roc_auc_score(y_valid, m.predict_proba(X_valid)[:, 1]),
        "test AUC": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]),
    })
comparison = pd.DataFrame(rows).set_index("model").round(3)
comparison["train-test gap"] = (comparison["train AUC"] - comparison["test AUC"]).round(3)
comparison

Read the table. All three land at a test AUC of about 0.79 to 0.80. The forest and XGBoost show a train-test gap of about 0.03 against roughly zero for logistic regression; that's the flexibility showing, and validation tuning kept it small. On survey data with eight well-behaved predictors and a smooth, monotone relationship (older and heavier means more diabetes), a straight line in log-odds space is already close to the best you can do. That is the usual result on tabular data of this kind, and it's why the simple model is the default and the complex one has to earn its place.

## 6. What the model learned

Three ways to look inside, from most to least direct:

1. **Coefficients** (logistic regression only): odds ratios, as in section 2.
2. **Feature importance** (trees): how much each feature reduced impurity across the trees. Fast, built in, and biased toward continuous features with many split points. Treat it as a ranking, not a measurement.
3. **Partial dependence**: hold everything else at its observed values, sweep one feature across its range, and plot the average predicted probability. Shows the *shape* of the relationship, which a coefficient can't.

In [ ]:
from model_metrics import get_feature_importances

feature_names = rf.estimator["preprocess_column_transformer_pre"].get_feature_names_out()
imp = pd.Series(get_feature_importances(rf.estimator, feature_names=feature_names)).sort_values(ascending=False)
imp.head(10).round(3)

In [ ]:
from model_metrics import plot_2d_pdp

# Partial dependence from XGBoost for the two features the tree split on first. X_train rows, feature names from the fitted preprocessor.
X_train_t = pd.DataFrame(xgb.estimator["preprocess_column_transformer_pre"].transform(X_train), columns=feature_names)
plot_2d_pdp(xgb.estimator["xgb"], X_train_t, feature_names=list(feature_names), features=["num__age", "num__bmi", "num__waist_cm"],
            title="Partial dependence (XGBoost), standardized units", grid_figsize=(14, 4), plot_type="grid")

Age dominates the importance ranking (0.45), then waist (0.21), then BMI (0.12). The partial dependence on age is the S-curve from chapter 1 again, and the model found it without being told the shape.

Now look at BMI and waist side by side: waist rises steadily across its range; BMI is nearly flat. The model didn't decide BMI doesn't matter. It had two features carrying the same information and gave the signal to one of them, and the PDP for the other shows what's left after that, which is nothing. This is the chapter 6 collinearity problem from the tree side: a partial dependence plot on one of two correlated features is only interpretable if you remember the other one exists. Drop BMI and the waist curve would look the same; drop waist and BMI would inherit the rise.

## 7. Choosing between models

| | Logistic regression | Random forest | Gradient boosting |
|---|---|---|---|
| Interpretability | odds ratios with CIs | importance ranking + PDPs | same |
| Captures non-linearity, interactions | only if you add them | automatically | automatically |
| Overfitting risk | low | moderate, averaging helps | higher, needs tuning |
| Tuning effort | one knob (C) | two or three | five or more |
| Typical gain over LR on tabular data | | small | small to moderate |
| When to prefer | explanation matters; data are smooth; you need to defend coefficients | quick strong baseline | squeezing the last few AUC points when prediction is all that matters |

For the clinic: logistic regression, with the odds ratios in the write-up, unless next chapter's evaluation shows the boosted model is meaningfully better *at the cutoff the clinic will actually use*. AUC is a ranking summary; the cutoff is where the decision lives.

## Looking ahead
- **Reading**: ISLR 4.4 (logistic regression) and chapter 8 (trees). Then ISLR 2.2.3 and 4.4.3 on the ROC curve, for next chapter.
- Next chapter: cutoffs, ROC and precision-recall, calibration (does 23% mean 23%?), and bootstrapped intervals on all of it.